In [ ]:
import os

import numpy as np

from rect_prune_ood.lib.util.paths import get_top_level_path

results_dir = os.path.join(get_top_level_path(), 'results')
imagenet_c_results_dir = os.path.join(results_dir, 'imagenet', "baseline", "imagenet_c")
ablation_study_results_dir = os.path.join(results_dir, 'ablation_study_pertubation')
os.makedirs(ablation_study_results_dir, exist_ok=True)

In [2]:
imagenet_c_pertubations = [
	"gaussian_noise",
	"saturate",
	"brightness",
	"pixelate",
	"snow",
	"motion_blur",
	"elastic_transform",
	"fog",
	"zoom_blur",
	"shot_noise",
	"contrast",
	"defocus_blur",
	"frost",
	"impulse_noise",
	"jpeg_compression",
	"glass_blur",
]

imagenet_c_pertubations_file = [
	"brightness",
	"defocus_blur",
	"fog",
	"gaussian_blur",
	"glass_blur",
	"jpeg_compression",
	"pixelate",
	"shot_noise",
	"spatter",
	"zoom_blur",
	"contrast",
	"elastic_transform",
	"frost",
	"gaussian_noise",
	"impulse_noise",
	"motion_blur",
	"saturate",
	"snow",
	"speckle_noise",
]

In [3]:
from torch import Tensor
from torch.nn import Module


class APESP(Module):
	def __init__(self, mask) -> None:
		# super(LayerScaler, self).__init__()
		super(APESP, self).__init__()
		self.mask = mask

	def forward(self, x: Tensor) -> Tensor:
		x = self.apes_p(x)
		return x

	def apes_p(self, x):
		assert x.dim() == 4
		_, c, _, _ = x.shape
		assert self.mask.shape[0] == c

		x = x * self.mask

		return x

In [4]:
import torch
from lightning import Trainer

from rect_prune_ood.lib.experiments.backbones import get_resnet50
from rect_prune_ood.lib.experiments.feature_extractor import FeatureExtractor
from rect_prune_ood.lib.experiments.pipeline import Pipeline
from rect_prune_ood.lib.util.data_loader import get_data_loader

imagenet_c_activations = np.load(
	os.path.join(imagenet_c_results_dir, "imagenet_c_activations_severity_5.npy")
)
imagenet_activations = np.load(
	os.path.join(results_dir, 'imagenet', 'baseline', 'imagenet_activations.npy')
)
imagenet_activations = np.transpose(imagenet_activations)

for i in range(len(imagenet_c_pertubations)):
	if i == 1:
		break

	pertubation_index = []
	for j in range(i + 1):
		pertubation_index.append(
			imagenet_c_pertubations_file.index(imagenet_c_pertubations[j])
		)

	# concatenate the activations. data has shape (19, 2048, 50000), goal is to get (2048, x)
	activations = np.concatenate(
		[imagenet_c_activations[j] for j in pertubation_index], axis=1
	)
	# concatenate with imagenet activations
	activations = np.concatenate([activations, imagenet_activations], axis=1)

	variance = np.var(activations, axis=1)
	percentile = np.percentile(variance, 95)

	mask = torch.zeros_like(torch.tensor(variance, dtype=torch.float32))
	mask[variance <= percentile] = 1
	mask = mask.view(variance.shape[0], 1, 1).cuda()

	feature_extractor = FeatureExtractor(get_resnet50(), 'avgpool')

	fc_layer = fc_layer = feature_extractor.get_fc_layer()

	pipeline = Pipeline(
		feature_extractor, 'baseline', fc_layer, caputure_activations=False
	)
	pipeline.method_list = [APESP(mask)]

	dataloader, _ = get_data_loader('imagenet', batch_size=16)

	# Trainer
	trainer = Trainer(logger=None)
	trainer.test(pipeline, dataloader)

	results = pipeline.results

	np.save(
		os.path.join(
			ablation_study_results_dir, f"ablation_study_pertubations_{i + 1}.npy"
		),
		results,
	)

FileNotFoundError: [Errno 2] No such file or directory: '/visinf/home/vilab16/RectPruneOOD/results/imagenet/baseline/imagenet_c/imagenet_c_activations_severity_5.npy'

In [ ]:
from rect_prune_ood.lib.eval.metrics import cal_metric

openimage_o_scores = np.load(
	os.path.join(results_dir, 'imagenet', 'apesp@95', 'openimage-o.npy'),
	allow_pickle=True,
).item()["energy_score"]

all_results = []

for i in range(len(imagenet_c_pertubations)):
	pertubation_scores = np.load(
		os.path.join(
			ablation_study_results_dir, f"ablation_study_pertubations_{i + 1}.npy"
		),
		allow_pickle=True,
	).item()["energy_score"]

	results = cal_metric(pertubation_scores, openimage_o_scores)
	results["num_pertubations"] = i + 1

	all_results.append(results)

print(all_results)

mtypes = ['FPR', 'AUROC']

print("num_p   FPR   AUROC", end='')

for results in all_results:
	print()
	print('{value:6d}'.format(value=results['num_pertubations']), end='')
	print('{value:6.2f}'.format(value=results['FPR']), end='')
	print('{value:6.2f}'.format(value=results['AUROC']), end='')

[{'FPR': 0.5757146098003629, 'AUROC': 0.8771103034255899, 'DTERR': 0.17605571327182398, 'AUIN': 0.9519118085323613, 'AUOUT': 0.7086729714276866, 'num_pertubations': 1}, {'FPR': 0.5763951905626135, 'AUROC': 0.8781987840290381, 'DTERR': 0.17574520936834634, 'AUIN': 0.9525169216102319, 'AUOUT': 0.7098601834103693, 'num_pertubations': 2}, {'FPR': 0.5719714156079855, 'AUROC': 0.8785898332577132, 'DTERR': 0.17445883605393897, 'AUIN': 0.9525483601477993, 'AUOUT': 0.7122654062877942, 'num_pertubations': 3}, {'FPR': 0.5695893829401089, 'AUROC': 0.8785587477313974, 'DTERR': 0.17494677075940382, 'AUIN': 0.9525476336390636, 'AUOUT': 0.7136167739106175, 'num_pertubations': 4}, {'FPR': 0.5640313067150635, 'AUROC': 0.88042291061706, 'DTERR': 0.17373432694582447, 'AUIN': 0.9533574259371542, 'AUOUT': 0.715995626975695, 'num_pertubations': 5}, {'FPR': 0.5593239564428312, 'AUROC': 0.8815589309210526, 'DTERR': 0.17246273953158267, 'AUIN': 0.9536939774194696, 'AUOUT': 0.7192945812464513, 'num_pertubations'